# Statistical Tests — 2025 Tariff Shock Option Microstructure

## What this notebook does

The EDA showed us *visually* that relative spreads widened during the tariff shock and that the widening differed across sectors. 

To determine if the underlying patterns occured by chance or not, we run four tests:

1. Did the shock significantly widen spreads?
2. Was the widening asymmetric across sectors?
3. Which option type spread widened more during the shock?
4. Did spreads fully recover?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import mannwhitneyu, kruskal
from itertools import combinations


PROJECT_ROOT = Path().resolve().parent
COMBINED_CSV = PROJECT_ROOT / "data" / "combined" / "combined_all.csv"

TICKERS      = ["AAPL", "NVDA", "AMZN", "PG", "CAT"]
COMMON_START = pd.Timestamp("2025-03-24")
ALPHA        = 0.05   # significance threshold


# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv(
    COMBINED_CSV,
    parse_dates=["collection_date", "expiration"],
    dtype={"ticker": str, "side": str, "moneyness_cat": str},
)
df["is_illiquid"] = df["is_illiquid"].astype(bool)

# Same subsets used in EDA -- keeping consistent means statistical results
# are directly comparable to the charts
liquid = df[~df["is_illiquid"]].copy()
atm    = liquid[liquid["moneyness_cat"] == "ATM"].copy()
cross  = atm[atm["collection_date"] >= COMMON_START].copy()

print(f"Full dataset : {len(df):,} rows")
print(f"ATM + liquid + common start: {len(cross):,} rows")
print()
print("Rows per phase:")
print(cross.groupby("phase").size().rename("count").to_string())

## Understanding the tools

### Why not a t-test?

The t-test is the most common way to compare two groups, but it assumes the data follows a **normal (bell curve) distribution**. Option bid-ask spreads are right-skewed — most contracts have moderate spreads, but a few have very wide spreads, pulling the tail to the right. A t-test on skewed data gives unreliable results.

The **Mann-Whitney U test** is the non-parametric alternative. Instead of comparing means, it asks: *if you randomly picked one observation from Group A and one from Group B, how often is the Group A value larger?* It works on rankings rather than raw values, so the skew doesn't matter.

### What is a p-value?

The p-value is the probability of seeing a difference this large (or larger) **purely by chance**, assuming there is actually no real difference. A small p-value means the observed pattern is unlikely to be noise.

- `p < 0.05` → statistically significant (we reject the idea that it's just chance)
- `p ≥ 0.05` → not significant (the data doesn't give us enough evidence)

The 0.05 threshold is a convention — it means we accept a 5% chance of a false positive.

### What is effect size?

Statistical significance tells you *whether* a difference exists. Effect size tells you *how big* it is. A result can be statistically significant but practically meaningless (e.g. spreads widened by 0.001% — real but irrelevant).

We use the **rank-biserial correlation** `r` as our effect size. It ranges from -1 to +1:
- `r ≈ 0` → no difference between groups
- `r > 0` → the second group tends to be larger
- `r < 0` → the first group tends to be larger

Magnitude thresholds (standard convention):
- `|r| < 0.1` → negligible
- `0.1 ≤ |r| < 0.3` → small
- `0.3 ≤ |r| < 0.5` → medium
- `|r| ≥ 0.5` → large

In [ ]:
# ── Shared helper functions used across all tests ─────────────────────────────

def rank_biserial_r(u_stat, n1, n2):
    """
    Compute rank-biserial correlation from a Mann-Whitney U statistic.

    Formula: r = 1 - (2 * U) / (n1 * n2)

    When mannwhitneyu(group1, group2) is called:
      - U is the number of pairs where a group1 value exceeds a group2 value
      - If U is small, group1 rarely beats group2, meaning group2 tends to be larger
      - r = 1 - 2*small/(n1*n2) → r close to +1 → group2 > group1

    So: positive r means group2 tends to be larger than group1.
    """
    return 1 - (2 * u_stat) / (n1 * n2)


def interpret_effect(r):
    """Classify the magnitude of a rank-biserial correlation."""
    a = abs(r)
    if a < 0.1:   return "negligible"
    elif a < 0.3: return "small"
    elif a < 0.5: return "medium"
    else:         return "large"


def significance_label(p, alpha=ALPHA):
    if p < 0.001:  return "*** (p<0.001)"
    elif p < 0.01: return "**  (p<0.01)"
    elif p < 0.05: return "*   (p<0.05)"
    else:          return "ns  (p≥0.05)"


print("Helper functions defined.")

---
## Test 1 — Did the shock significantly widen spreads?

**Comparison:** Phase 1 (baseline) vs Phase 2 (shock) relative spread, per ticker

**Test:** Mann-Whitney U (two-sided)

**What we expect:** The EDA charts showed spread spikes around Apr 2–9. This test checks whether those spikes are statistically distinguishable from normal Phase 1 variation — i.e., whether the entire Phase 2 spread distribution shifted up, not just a few extreme days.

A two-sided test is used because we don't want to assume in advance that spreads can only widen. The data should tell us the direction.

In [ ]:
print("Test 1: Phase 1 vs Phase 2 relative spread (ATM liquid, per ticker)")
print("=" * 70)

results_t1 = []

for ticker in TICKERS:
    # Extract phase 1 and phase 2 spreads for this ticker
    s1 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 1), "relative_spread"].dropna()
    s2 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 2), "relative_spread"].dropna()

    # Run the Mann-Whitney U test
    # alternative="two-sided" means we test whether the two distributions differ
    # in either direction, not just whether phase2 > phase1
    u_stat, p = mannwhitneyu(s1, s2, alternative="two-sided")

    # Effect size
    r = rank_biserial_r(u_stat, len(s1), len(s2))

    pct_change = (s2.median() - s1.median()) / s1.median() * 100

    results_t1.append({
        "Ticker":         ticker,
        "N Phase1":       len(s1),
        "N Phase2":       len(s2),
        "Median P1":      round(s1.median(), 4),
        "Median P2":      round(s2.median(), 4),
        "Change %":       round(pct_change, 1),
        "p-value":        round(p, 5),
        "Significance":   significance_label(p),
        "Effect r":       round(r, 3),
        "Magnitude":      interpret_effect(r),
    })

t1 = pd.DataFrame(results_t1).set_index("Ticker")
print(t1.to_string())
print()
print("r > 0 means Phase 2 spreads tended to be LARGER than Phase 1 (widening).")

---
## Test 2 — Was the widening asymmetric across sectors?

Test 1 confirmed whether spreads widened for each ticker individually. Test 2 asks whether the **magnitude of widening differed across sectors** — i.e., did the market treat some sectors as more impacted than others?

### Step 2a: Kruskal-Wallis test

The **Kruskal-Wallis** test is the multi-group extension of Mann-Whitney. Instead of comparing two groups, it compares K groups simultaneously and tests whether at least one group has a different distribution.

- **Null hypothesis:** all five tickers have the same Phase 2 relative spread distribution
- **Alternative:** at least one ticker differs

If Kruskal-Wallis is significant, we know there is asymmetry somewhere — but not which specific pairs differ. For that we need Step 2b.

**Effect size:** We use **eta-squared (η²)**, which measures what fraction of the total variance in spreads is explained by which ticker it is.
- η² = 0.01 → small
- η² = 0.06 → medium  
- η² = 0.14 → large

In [ ]:
print("Test 2a: Kruskal-Wallis — are Phase 2 spreads different across tickers?")
print("=" * 70)

phase2 = cross[cross["phase"] == 2]

# Build one array of spreads per ticker
groups = [
    phase2.loc[phase2["ticker"] == t, "relative_spread"].dropna().values
    for t in TICKERS
]

# Print group sizes and medians so we know what we're comparing
print("Phase 2 median relative spread per ticker:")
for ticker, g in zip(TICKERS, groups):
    print(f"  {ticker}: n={len(g):,}  median={np.median(g):.4f}")
print()

# Run the Kruskal-Wallis test
# *groups unpacks the list so kruskal receives each ticker's array as a separate argument
h_stat, p_kw = kruskal(*groups)

# Eta-squared effect size for Kruskal-Wallis:
# eta_sq = (H - k + 1) / (N - k)
# where H is the test statistic, k is the number of groups, N is total observations
k = len(TICKERS)
N = sum(len(g) for g in groups)
eta_sq = (h_stat - k + 1) / (N - k)

print(f"Kruskal-Wallis H = {h_stat:.3f}")
print(f"p-value          = {p_kw:.6f}  {significance_label(p_kw)}")
print(f"Eta-squared (η²) = {eta_sq:.4f}")
print()
if p_kw < ALPHA:
    print("Result: At least one sector had a significantly different spread during Phase 2.")
    print("        Proceed to pairwise comparisons to find which pairs differ.")
else:
    print("Result: No significant difference in spreads across sectors during Phase 2.")

### Step 2b: Pairwise comparisons with Bonferroni correction

Kruskal-Wallis tells us *something* differs but not *what*. We now run Mann-Whitney for every pair of tickers (C(5,2) = 10 pairs).

**The multiple comparisons problem:** If you run 10 tests each at α = 0.05, you have a 1 - (0.95)¹⁰ ≈ 40% chance of getting at least one false positive just by chance — even if nothing is actually different. Running more tests inflates your false positive rate.

**Bonferroni correction** is the simplest fix: divide your significance threshold by the number of tests. With 10 pairs and α = 0.05, the adjusted threshold is **α = 0.005**. Equivalently, multiply each raw p-value by 10 and compare to the original 0.05 — that's what we do below.

The correction is conservative (it may miss some real differences), but it's the right starting point.

In [ ]:
print("Test 2b: Pairwise Mann-Whitney with Bonferroni correction")
print("=" * 70)

pairs = list(combinations(TICKERS, 2))
n_comparisons = len(pairs)
bonferroni_alpha = ALPHA / n_comparisons

print(f"Number of pairs: {n_comparisons}")
print(f"Bonferroni-adjusted alpha: {ALPHA} / {n_comparisons} = {bonferroni_alpha:.4f}")
print()

pairwise_results = []

for t1, t2 in pairs:
    s1 = phase2.loc[phase2["ticker"] == t1, "relative_spread"].dropna()
    s2 = phase2.loc[phase2["ticker"] == t2, "relative_spread"].dropna()

    u_stat, p_raw = mannwhitneyu(s1, s2, alternative="two-sided")

    # Bonferroni adjustment: multiply raw p by number of comparisons
    # Clamp at 1.0 since a probability cannot exceed 1
    p_adj = min(p_raw * n_comparisons, 1.0)

    r = rank_biserial_r(u_stat, len(s1), len(s2))

    pairwise_results.append({
        "Pair":          f"{t1} vs {t2}",
        "Med(t1)": round(s1.median(), 4),
        "Med(t2)": round(s2.median(), 4),
        "p (raw)":       round(p_raw, 5),
        "p (Bonferroni)": round(p_adj, 4),
        "Significant":   "Yes" if p_adj < ALPHA else "No",
        "Effect r":      round(r, 3),
        "Magnitude":     interpret_effect(r),
    })

t2b = pd.DataFrame(pairwise_results).set_index("Pair")
print(t2b.to_string())
print()
sig_pairs = t2b[t2b["Significant"] == "Yes"]
print(f"Significant pairs after Bonferroni correction: {len(sig_pairs)} / {n_comparisons}")

---
## Test 3 — Did put spreads widen more than call spreads?

In normal markets, put and call spreads for the same strike and expiry are roughly equal — put-call parity keeps them anchored. During a shock, demand for **downside protection surges** (traders buy puts to hedge). This flood of demand overwhelms market makers' ability to quote tight markets, so **put spreads widen more than call spreads**.

A significant difference between put and call spreads during Phase 2 is evidence that the market was pricing in directional fear, not just general uncertainty.

**What to expect:** puts wider than calls (r < 0, since we put calls first and puts second in the test).

In [ ]:
print("Test 3: Call vs Put relative spread during Phase 2 (ATM liquid, per ticker)")
print("=" * 70)

results_t3 = []

for ticker in TICKERS:
    subset = cross[(cross["ticker"] == ticker) & (cross["phase"] == 2)]

    calls = subset.loc[subset["side"] == "call", "relative_spread"].dropna()
    puts  = subset.loc[subset["side"] == "put",  "relative_spread"].dropna()

    u_stat, p = mannwhitneyu(calls, puts, alternative="two-sided")

    # r > 0 means puts > calls (since puts is the second argument)
    r = rank_biserial_r(u_stat, len(calls), len(puts))

    results_t3.append({
        "Ticker":        ticker,
        "N Calls":       len(calls),
        "N Puts":        len(puts),
        "Median Call":   round(calls.median(), 4),
        "Median Put":    round(puts.median(), 4),
        "Put > Call":    puts.median() > calls.median(),
        "p-value":       round(p, 5),
        "Significance":  significance_label(p),
        "Effect r":      round(r, 3),
        "Magnitude":     interpret_effect(r),
    })

t3 = pd.DataFrame(results_t3).set_index("Ticker")
print(t3.to_string())
print()
print("r > 0 means put spreads tended to be WIDER than call spreads.")

---
## Test 4 — Did spreads fully recover?

**Comparison:** Phase 1 (baseline) vs Phase 3 (recovery) relative spread, per ticker

If spreads **fully recovered**, we expect:
- No significant difference between Phase 1 and Phase 3
- Effect size close to zero

If spreads **did not recover**, we expect:
- Significant difference (Phase 3 still wider than Phase 1)
- Positive effect r

This is analytically interesting because it distinguishes sectors the market treated as **temporarily disrupted** (recover quickly) from those seen as **structurally affected** (slow or incomplete recovery). That's one of the core claims of this project.

In [ ]:
print("Test 4: Phase 1 vs Phase 3 relative spread (ATM liquid, per ticker)")
print("=" * 70)

results_t4 = []

for ticker in TICKERS:
    s1 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 1), "relative_spread"].dropna()
    s3 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 3), "relative_spread"].dropna()

    u_stat, p = mannwhitneyu(s1, s3, alternative="two-sided")
    r = rank_biserial_r(u_stat, len(s1), len(s3))

    pct_change = (s3.median() - s1.median()) / s1.median() * 100

    # Not significant = spreads are statistically indistinguishable from baseline = recovered
    recovered = p >= ALPHA

    results_t4.append({
        "Ticker":        ticker,
        "Median P1":     round(s1.median(), 4),
        "Median P3":     round(s3.median(), 4),
        "Change %":      round(pct_change, 1),
        "p-value":       round(p, 5),
        "Significance":  significance_label(p),
        "Effect r":      round(r, 3),
        "Magnitude":     interpret_effect(r),
        "Recovered?": "Yes" if recovered else "No",
    })

t4 = pd.DataFrame(results_t4).set_index("Ticker")
print(t4.to_string())
print()
print("'Recovered = Yes' means Phase 3 spreads are statistically indistinguishable from Phase 1.")
print("r > 0 means Phase 3 spreads are still wider than Phase 1 (incomplete recovery).")

---
## Summary of All Findings

In [ ]:
print("SUMMARY")
print("=" * 70)
print()

print("Test 1: Did the shock significantly widen spreads? (Phase 1 vs Phase 2)")
print("-" * 50)
for _, row in t1.iterrows():
    print(f"  {row.name}: {row['Significance']}  |  "
          f"+{row['Change %']}%  |  "
          f"effect r={row['Effect r']} ({row['Magnitude']})")

print()
print("Test 2a: Kruskal-Wallis — asymmetry across sectors during Phase 2")
print("-" * 50)
print(f"  H={h_stat:.3f}, p={p_kw:.5f} {significance_label(p_kw)}, η²={eta_sq:.4f}")

print()
print("Test 2b: Pairwise comparisons (Bonferroni corrected)")
print("-" * 50)
for pair, row in t2b.iterrows():
    star = "*" if row["Significant"] == "Yes" else " "
    print(f"  {star} {pair}: p_adj={row['p (Bonferroni)']:.4f}  |  "
          f"effect r={row['Effect r']} ({row['Magnitude']})")

print()
print("Test 3: Put vs Call spread asymmetry during Phase 2")
print("-" * 50)
for _, row in t3.iterrows():
    direction = "puts wider" if row["Put > Call"] else "calls wider"
    print(f"  {row.name}: {row['Significance']}  |  "
          f"{direction}  |  effect r={row['Effect r']} ({row['Magnitude']})")

print()
print("Test 4: Did spreads recover? (Phase 3 vs Phase 1)")
print("-" * 50)
for _, row in t4.iterrows():
    print(f"  {row.name}: Recovered={row['Recovered?']}  |  "
          f"{row['Change %']:+.1f}% vs baseline  |  "
          f"{row['Significance']}")